<a href="https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

The purpose of this playbook is to turn the validated model output into a ranked review queue for content teams.

The model produces a probability that a content item belongs to the declining-label class. I use this probability to rank items for review rather than treating it as a guaranteed prediction of future performance.

Each ranked item receives a reason code based on observable characteristics in the data. The reason codes are intended to make the queue easier for a human reviewer to interpret.

The main actions are:

* **DECLINE_RISK:** the model assigns a relatively high declining-label probability, so the page is prioritized for review.
* **AGE_AND_DECLINE:** the page has a decline signal and is also relatively old, making content freshness worth checking.
* **PERFORMANCE_REVIEW:** the page has a decline signal and measurable performance history that should be inspected by a reviewer.
* **MONITOR:** the model signal is weaker, so the item is not prioritized for immediate action.

These codes are recommendations for review, not automatic decisions. A high model score does not establish that changing a page will improve its future performance.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 — Section 1: Ranked actions + reason codes

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Load starter dataset
DATA_URL = (
    "https://raw.githubusercontent.com/ritakimani9-lang/"
    "machinelearning/main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

# Recreate the target exactly as in W05/W06
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# Exact W05 model feature set
NUM_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "days_with_impressions",
    "days_with_sessions",
]

CAT_FEATURES = [
    "content_type",
    "main_intent",
    "competition_level",
]

FEATURES = NUM_FEATURES + CAT_FEATURES

X = df[FEATURES].copy()
y = df["is_declining_label"].copy()

# Same preprocessing/model family used in W05
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUM_FEATURES),
    ("cat", categorical_pipe, CAT_FEATURES),
])

model = RandomForestClassifier(
    300,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

# Fit on the full starter dataset for queue generation.
# This queue is decision-support output, not a sealed test result.
pipeline.fit(X, y)

# Generate declining-label probabilities
df["decline_probability"] = pipeline.predict_proba(X)[:, 1]

# Rank highest probability first
queue = df.sort_values(
    "decline_probability",
    ascending=False
).reset_index(drop=True)

queue["priority_rank"] = np.arange(1, len(queue) + 1)

# Relative age signal for reason coding
age_median = df["content_age_days"].median()

def assign_reason(row):
    if row["decline_probability"] >= 0.75 and row["content_age_days"] >= age_median:
        return "AGE_AND_DECLINE"
    elif row["decline_probability"] >= 0.75:
        return "DECLINE_RISK"
    elif row["decline_probability"] >= 0.60:
        return "PERFORMANCE_REVIEW"
    else:
        return "MONITOR"

queue["reason_code"] = queue.apply(assign_reason, axis=1)

# Suggested human action
action_map = {
    "AGE_AND_DECLINE": "Review freshness, search performance, and content quality.",
    "DECLINE_RISK": "Review page performance and investigate possible causes.",
    "PERFORMANCE_REVIEW": "Inspect search and engagement signals before deciding on action.",
    "MONITOR": "Monitor rather than prioritize for immediate content changes.",
}

queue["suggested_action"] = queue["reason_code"].map(action_map)

# Display only public-safe fields
display_columns = [
    "priority_rank",
    "decline_probability",
    "reason_code",
    "suggested_action",
    "content_type",
    "main_intent",
    "content_age_days",
    "days_since_last_update",
]

display(queue[display_columns].head(20))

print("Queue rows:", len(queue))
print("Declining-label base rate:", round(y.mean(), 3))

,priority_rank,decline_probability,reason_code,suggested_action,content_type,main_intent,content_age_days,days_since_last_update
0,1,0.974477,DECLINE_RISK,Review page performance and investigate possib...,keyword article,informational,147,20
1,2,0.968170,DECLINE_RISK,Review page performance and investigate possib...,keyword article,informational,147,20
2,3,0.964711,DECLINE_RISK,Review page performance and investigate possib...,keyword article,informational,141,20
3,4,0.963346,DECLINE_RISK,Review page performance and investigate possib...,keyword article,informational,141,20
4,5,0.963179,DECLINE_RISK,Review page performance and investigate possib...,keyword article,informational,147,20
5,6,0.961092,DECLINE_RISK,Review page performance and investigate possib...,keyword article,informational,141,20
6,7,0.960047,DECLINE_RISK,Review page performance and investigate possib...,keyword article,informational,147,20
7,8,0.959452,DECLINE_RISK,Review page performance and investigate possib...,keyword article,informational,141,20
8,9,0.958150,DECLINE_RISK,Review page performance and investigate possib...,keyword article,informational,141,20
9,10,0.958148,DECLINE_RISK,Review page performance and investigate possib...,keyword article,informational,147,20


Queue rows: 30000
Declining-label base rate: 0.542


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

The intended use of this playbook is to help a content team prioritize which pages deserve human review first.

The model ranks pages according to their measured association with the declining label. The queue is therefore a triage tool rather than an automated content-management system.

A reviewer can use the reason code and supporting page characteristics to decide what to investigate. Possible investigations include content freshness, search performance, engagement, and whether the page still matches the intended search intent.

There are several important limits.

First, the evaluation in W06 used client-grouped validation across 32 clients, so the evidence is limited to this dataset and its available client population.

Second, the current feature set contains 90-day metrics that overlap the period used to construct the declining label. Grouped validation does not remove this same-window overlap.

Third, the queue is generated using the full starter dataset for prioritization and should not be interpreted as a new unbiased test result.

Finally, the model identifies pages worth reviewing; it does not establish that a particular content change will cause performance to improve.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 — Section 2: Intended-use checks

print("INTENDED USE")
print("Purpose: prioritize content pages for human review.")
print("Output: ranked review queue with reason codes.")
print("Automation level: decision support only.")

print("\nKEY LIMITATIONS")
print("1. 32-client starter dataset.")
print("2. Current 90-day features overlap the label period.")
print("3. Queue probabilities are not future guarantees.")
print("4. Full-data fit is used only to generate the review queue.")
print("5. Content changes remain human decisions.")

INTENDED USE
Purpose: prioritize content pages for human review.
Output: ranked review queue with reason codes.
Automation level: decision support only.

KEY LIMITATIONS
1. 32-client starter dataset.
2. Current 90-day features overlap the label period.
3. Queue probabilities are not future guarantees.
4. Full-data fit is used only to generate the review queue.
5. Content changes remain human decisions.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review + the no-go list

Every item in the queue requires human review before an action is taken.

The reviewer should first check whether the page has enough usable data to support an investigation. The reviewer should then inspect the reason code together with relevant page context, including content age, recent update history, search performance, engagement signals, and search intent.

The reviewer should also check whether there are known business or editorial reasons that make the recommendation inappropriate.

### Human-review rules

1. Treat the model score as a prioritization signal, not a guaranteed outcome.
2. Check the underlying page and its business/search context before changing content.
3. Check whether the data appears complete enough to support the recommendation.
4. Do not interpret correlation or association as proof that a content change will improve performance.
5. Record the human decision and reason for any action taken.
6. If the evidence is unclear, leave the page for monitoring rather than forcing an intervention.

### No-go cases

The following should not be automated:

* publishing or deleting content;
* rewriting a page solely because of its model score;
* changing search intent or keyword targeting without human review;
* changing business-critical pages automatically;
* declaring that a content change caused an improvement without an appropriate evaluation design;
* treating the model score as a guarantee of future traffic or ranking;
* overriding editorial, legal, brand, or business requirements automatically.

The playbook therefore keeps the final content decision with a human reviewer.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 — Section 3: Human-review / no-go checks

NO_GO_AUTOMATIONS = [
    "automatic publishing",
    "automatic deletion",
    "automatic rewriting",
    "automatic search-intent changes",
    "automatic changes to business-critical pages",
    "automatic causal attribution",
]

print("Human review required: YES")
print("\nActions explicitly kept out of automation:")

for item in NO_GO_AUTOMATIONS:
    print("-", item)

Human review required: YES

Actions explicitly kept out of automation:
- automatic publishing
- automatic deletion
- automatic rewriting
- automatic search-intent changes
- automatic changes to business-critical pages
- automatic causal attribution


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring / retrain triggers

The playbook should be monitored because the relationship between available signals and the declining label may change over time.

The following checks are useful monitoring triggers:

* **Base-rate drift:** the proportion of pages receiving the declining label changes substantially from the development population.
* **Feature drift:** important input distributions change noticeably from the data used to build the model.
* **Performance drift:** Precision@100 or ROC-AUC falls below the previously measured validation range when evaluated on newly labeled data.
* **Coverage changes:** important fields become substantially more missing or unavailable.
* **New client or content populations:** the model is applied to populations that differ materially from the development data.
* **Data-window changes:** the definition of the outcome or available history changes.
* **Repeated human disagreement:** reviewers frequently reject high-priority recommendations for the same reason.

A retrain should not be triggered by a single unusual prediction. Instead, monitoring should look for sustained changes in data quality, population, or measured performance.

Retraining should also be followed by the same leakage and grouped/time-aware validation checks used during development. A new model should not automatically replace the existing one simply because it produces a different score.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 — Section 4: Current monitoring baseline

monitoring_baseline = {
    "rows": len(df),
    "clients": df["client_id"].nunique(),
    "label_base_rate": y.mean(),
    "median_content_age_days": df["content_age_days"].median(),
    "median_days_since_last_update": df["days_since_last_update"].median(),
}

print("CURRENT DEVELOPMENT BASELINE")
for key, value in monitoring_baseline.items():
    if isinstance(value, float):
        print(f"{key}: {value:.3f}")
    else:
        print(f"{key}: {value}")

print("\nSuggested monitoring triggers:")
print("- label/base-rate drift")
print("- feature distribution drift")
print("- missingness/coverage drift")
print("- measured Precision@100 or ROC-AUC deterioration")
print("- population/client changes")
print("- repeated human rejection of recommendations")

CURRENT DEVELOPMENT BASELINE
rows: 30000
clients: 32
label_base_rate: 0.542
median_content_age_days: 236.000
median_days_since_last_update: 20.000

Suggested monitoring triggers:
- label/base-rate drift
- feature distribution drift
- missingness/coverage drift
- measured Precision@100 or ROC-AUC deterioration
- population/client changes
- repeated human rejection of recommendations


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The ranked queue is exported to `work/outputs/` so that the research paper can reuse the same action-playbook output.

The export contains the ranking, model signal, reason code, suggested action, and selected supporting fields needed for interpretation.

The queue is regenerated by the notebook rather than committed as a data file. This keeps the repository free of the underlying content-level data while preserving a reproducible process for producing the output.

The exported queue should be interpreted as decision-support evidence rather than as a production recommendation system.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 — Section 5: Export queue

from pathlib import Path

output_dir = Path("../../work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Keep export public-safe and useful for the paper.
export_columns = [
    "priority_rank",
    "decline_probability",
    "reason_code",
    "suggested_action",
    "content_type",
    "main_intent",
    "competition_level",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

queue_export = queue[export_columns].copy()

queue_path = output_dir / "w07_ranked_action_queue.csv"
queue_export.to_csv(queue_path, index=False)

print(f"Exported: {queue_path}")
print(f"Rows exported: {len(queue_export):,}")

# Verification
assert queue_path.exists()
assert len(queue_export) == len(queue)

print("Export verification: PASS")

Exported: ../../work/outputs/w07_ranked_action_queue.csv
Rows exported: 30,000
Export verification: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.